# 05 — Knowledge Graph

Drug-disease-comorbidity knowledge graph for the T2DM Persistence RWE study. Visualises TREATS, ASSOCIATED_WITH, and DRUG_CLASS_EFFECT relationships.

In [1]:
import sys
sys.path.insert(0, '../..')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
from graph.build_graph import build_graph

G = build_graph(
    cohort_path='../../outputs/tables/cohort_matched.csv',
    comorbidity_path='../../cohort/codx_mapping.xlsx',
    output_dir='../../graph/cypher_export',
    corr_path='../../outputs/tables/correlations.csv',
    cox_ttc_path='../../outputs/tables/cox_ttc_results.csv',
)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

2026-05-12 18:06:22,267 INFO Graph: 19 nodes, 27 edges


2026-05-12 18:06:22,573 INFO Graph visualisation saved: outputs/figures/knowledge_graph.png


2026-05-12 18:06:22,574 INFO Cypher files exported: nodes.cypher, edges.cypher


Graph: 19 nodes, 27 edges


In [2]:
# Node type summary
from collections import Counter
type_counts = Counter(nx.get_node_attributes(G, 'node_type').values())
print('Node types:', dict(type_counts))

edge_relations = Counter(d.get('relation', 'unknown') for _, _, d in G.edges(data=True))
print('Edge relations:', dict(edge_relations))

Node types: {'DrugClass': 3, 'Comorbidity': 15, 'Outcome': 1}
Edge relations: {'TREATS': 11, 'DRUG_CLASS_EFFECT': 2, 'ASSOCIATED_WITH': 14}


In [3]:
# Display the saved graph PNG
fig, ax = plt.subplots(figsize=(10, 8))
img = plt.imread('../../outputs/figures/knowledge_graph.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('T2DM Drug-Disease-Comorbidity Knowledge Graph')
plt.tight_layout()
plt.show()

/var/folders/5_/fn5m9ggx60b349bz6bzp1qcc0000gn/T/ipykernel_55961/2354918481.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Degree centrality — most connected nodes
centrality = nx.degree_centrality(G)
top_nodes = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print('Top 10 nodes by degree centrality:')
for node, cent in top_nodes:
    label = G.nodes[node].get('label', node)
    print(f'  {label:<30} {cent:.3f}')

Top 10 nodes by degree centrality:
  Treatment
Discontinuation      0.889
  GLP-1 RA                       0.333
  SGLT-2i                        0.278
  Obesity                        0.167
  Ckd                            0.167
  Heart Failure                  0.167
  Mi                             0.167
  Metformin                      0.111
  Hyperlipidemia                 0.111
  Pvd                            0.111


In [5]:
# Show Cypher snippet
from pathlib import Path
nodes_cyp = Path('../../graph/cypher_export/nodes.cypher').read_text()
print('First 5 node Cypher statements:')
print('\n'.join(nodes_cyp.splitlines()[:5]))

First 5 node Cypher statements:
MERGE (n:DrugClass {id: 'metformin', name: 'Metformin'});
MERGE (n:DrugClass {id: 'glp1', name: 'GLP-1 RA'});
MERGE (n:DrugClass {id: 'sglt2', name: 'SGLT-2i'});
MERGE (n:Comorbidity {id: 'hypertension', name: 'Hypertension'});
MERGE (n:Comorbidity {id: 'obesity', name: 'Obesity'});
